# `run_pipeline.py`

Notebook version of `run_pipeline.py`. The `.py` file is the source of truth (the app and the other scripts import it); after editing it, regenerate this notebook with `python tools/py_to_ipynb.py run_pipeline.py --overwrite`.

```text
run_pipeline.py - build everything from the raw spreadsheet, in order.

  python run_pipeline.py            # ETL, knowledge index, embeddings, property models, tests
  python run_pipeline.py --eval     # ... and the full six-model evaluation (1.5-2 h, mostly free-tier waits)
```

In [ ]:
# Notebook setup: make the project root importable and define __file__,
# so the code behaves as if run as a script (run Jupyter from this file's folder).
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
__file__ = str(ROOT / 'run_pipeline.py')

In [ ]:
import subprocess

import sys

STEPS = [
    ("Build SQLite database", ["setup_db.py"]),
    ("Build knowledge (RAG) index", ["setup_vectorstore.py"]),
    ("Train FT-Transformer embeddings", ["train_ft.py"]),
    ("Train property models (+SHAP)", ["train_property_models.py"]),
    ("Unit tests", ["-m", "pytest", "-q"]),
]

In [ ]:
if __name__ == "__main__":
    steps = STEPS + ([("Evaluation", ["evaluate.py"])] if "--eval" in sys.argv else [])
    for name, args in steps:
        print(f"\n=== {name} ===", flush=True)
        if subprocess.run([sys.executable, *args]).returncode != 0:
            sys.exit(f"step failed: {name}")
    print("\nDone. Start the app with:  streamlit run app.py")